# Infrared City UTCI Test

This notebook runs a simple UTCI comparison for the same area covered by the Vienna orthofoto image.

It uses `outputs/detected_trees.geojson` as detected vegetation input and compares a run without trees to a run with detected trees.

The workflow follows the Infrared UTCI demo pattern: fetch buildings and ground materials, find a nearby weather station, filter weather data, build a UTCI payload, and run `client.run_area_and_wait()`.

In [ ]:
# Import the libraries we need.
# os lets us read environment variables like INFRARED_API_KEY.
# sys lets us add the project folder to Python's import path.
# Path helps us build file paths that work on different operating systems.
# pprint prints dictionaries in a readable way.
import json
import os
import sys
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

In [ ]:
# Find the project root folder.
# If this notebook is opened from the notebooks/ folder, the project root is one level up.
# If it is opened from the project root, the current folder is already the project root.
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# Add the project root to Python's import path.
# This lets the notebook import code from backend/app/simulation.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load variables from the .env file into Python's environment.
load_dotenv(project_root / ".env", override=True)

In [ ]:
# Load the Infrared API key.
# We only print whether it exists, not the key itself.
infrared_api_key = os.getenv("INFRARED_API_KEY")

if not infrared_api_key:
    raise ValueError("INFRARED_API_KEY is missing. Add it to your .env file first.")

print("INFRARED_API_KEY loaded.")

In [ ]:
# Import the reusable Infrared test helpers from the backend.
# Reload the module so Jupyter picks up recent edits without requiring a kernel restart.
import importlib
import backend.app.simulation.infrared_runner as infrared_runner

infrared_runner = importlib.reload(infrared_runner)

load_tree_geojson = infrared_runner.load_tree_geojson
run_utci_comparison = infrared_runner.run_utci_comparison
save_utci_heatmap = infrared_runner.save_utci_heatmap
save_utci_outputs = infrared_runner.save_utci_outputs
save_vegetation_geojson = infrared_runner.save_vegetation_geojson
tree_geojson_to_canopy_polygons = infrared_runner.tree_geojson_to_canopy_polygons
tree_geojson_to_infrared_vegetation = infrared_runner.tree_geojson_to_infrared_vegetation
from backend.app.imagery.vienna_orthofoto import create_bbox_polygon
from backend.app.visualization.input_maps import plot_utci_grids

In [ ]:
# Load detected tree GeoJSON from Step 03 and Vienna orthofoto metadata from Step 01b.
tree_geojson_path = project_root / "outputs" / "detected_trees.geojson"
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing Vienna orthofoto metadata: {metadata_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
default_center_lat = metadata["center"]["lat"]
default_center_lon = metadata["center"]["lon"]
default_zoom = metadata["zoom"]
default_image_width = metadata["image_width"]
default_image_height = metadata["image_height"]

tree_geojson = load_tree_geojson(str(tree_geojson_path))
tree_metadata = tree_geojson.get("metadata", {})
if tree_metadata.get("center") != metadata.get("center") or tree_metadata.get("bbox") != metadata.get("bbox"):
    raise ValueError(
        "Detected tree GeoJSON metadata does not match the current Vienna orthofoto metadata. "
        "Rerun notebooks 02 and 03 after changing notebook 01b coordinates."
    )

vegetation = tree_geojson_to_infrared_vegetation(tree_geojson)
canopy_polygons = tree_geojson_to_canopy_polygons(tree_geojson)

# Use Infrared point-tree features for the live UTCI payload.
# The canopy polygons are saved for visualization, but the current Infrared UTCI API expects point vegetation.
vegetation_geometry_for_utci = "points"
utci_vegetation = canopy_polygons if vegetation_geometry_for_utci == "canopy_polygons" else vegetation

infrared_points_geojson_path = project_root / "outputs" / "infrared_tree_points.geojson"
canopy_geojson_path = project_root / "outputs" / "infrared_tree_canopies.geojson"
utci_payload_geojson_path = project_root / "outputs" / "infrared_tree_payload.geojson"
saved_infrared_points_geojson = save_vegetation_geojson(vegetation, str(infrared_points_geojson_path))
saved_canopy_geojson = save_vegetation_geojson(canopy_polygons, str(canopy_geojson_path))
saved_utci_payload_geojson = save_vegetation_geojson(utci_vegetation, str(utci_payload_geojson_path))

canopy_radii = [
    feature.get("properties", {}).get("canopy_radius_m")
    for feature in tree_geojson.get("features", [])
    if feature.get("properties", {}).get("canopy_radius_m") is not None
]

print(f"Loaded detected tree features: {len(tree_geojson.get('features', []))}")
print(f"Prepared Infrared point tree features: {len(vegetation)}")
print(f"Prepared canopy polygon features: {len(canopy_polygons)}")
print(f"UTCI vegetation geometry: {vegetation_geometry_for_utci}")
print(f"Saved point-tree payload to: {saved_infrared_points_geojson}")
print(f"Saved canopy polygons to: {saved_canopy_geojson}")
print(f"Saved actual UTCI vegetation payload to: {saved_utci_payload_geojson}")
payload_geometry_types = sorted({feature.get("geometry", {}).get("type") for feature in utci_vegetation.values()})
print(f"Actual UTCI payload geometry types: {payload_geometry_types}")
if canopy_radii:
    print(f"Average estimated canopy radius: {sum(canopy_radii) / len(canopy_radii):.2f} m")
    print(f"Max estimated canopy radius: {max(canopy_radii):.2f} m")

print(f"Vienna orthofoto center: lat={default_center_lat}, lon={default_center_lon}")

In [ ]:
# Create the UTCI test polygon from the exact Vienna orthofoto lon/lat bounds.
# GeoJSON coordinates use [longitude, latitude] order.
test_polygon = create_bbox_polygon(metadata["bbox_lonlat"])

print("Image-bounds simulation polygon:")
pprint(test_polygon)

In [ ]:
# Run the UTCI comparison.
# This calls the live Infrared SDK, so it may take time and may consume account credits.
# Set run_live=False if you only want to inspect the prepared payload summary.
# TODO: Confirm the desired UTCI time period and tree canopy assumptions.
run_live = True

result = run_utci_comparison(
    tree_geojson_path=str(tree_geojson_path),
    center_lon=default_center_lon,
    center_lat=default_center_lat,
    zoom=default_zoom,
    image_width=default_image_width,
    image_height=default_image_height,
    run_live=run_live,
    polygon=test_polygon,
    vegetation_geometry=vegetation_geometry_for_utci,
    input_metadata=metadata,
)

print(f"Simulation status: {result['status']}")
print(f"Weather file ID: {result.get('weather_file_id')}")
print(f"Building count: {result.get('building_count')}")
print(f"Ground material feature count: {result.get('ground_material_feature_count')}")
print(f"Detected tree count: {result.get('detected_tree_count')}")
print(f"Average UTCI without trees: {result['average_utci_without_trees']}")
print(f"Average UTCI with trees: {result['average_utci_with_trees']}")
print(f"UTCI difference: {result['utci_difference']}")

In [ ]:
# Save useful UTCI outputs for later notebooks or the backend.
# If the live SDK call did not complete, this still writes a summary JSON,
# but it marks the result as placeholder and keeps numeric UTCI fields as null.
saved_outputs = save_utci_outputs(
    result=result,
    output_dir=str(project_root / "outputs"),
)

print(f"Saved UTCI summary JSON to: {saved_outputs['summary_path']}")
print(f"Saved UTCI without-trees grid to: {saved_outputs['utci_without_trees_npy']}")
print(f"Saved UTCI with-trees grid to: {saved_outputs['utci_with_trees_npy']}")
pprint(saved_outputs["summary"])

if (
    saved_outputs["summary"].get("status") == "completed"
    and saved_outputs["summary"].get("vegetation_feature_count_with", 0) > 0
    and saved_outputs["summary"].get("max_abs_utci_difference") == 0
):
    raise RuntimeError(
        "Infrared completed but the with-tree UTCI grid is identical to the without-tree grid. "
        "This means vegetation was not integrated into the UTCI result. Check outputs/infrared_tree_payload.geojson "
        "and rerun with vegetation_geometry_for_utci='points'."
    )

In [ ]:
# Plot UTCI outputs directly inside the notebook if real grids were saved.
# This does not fake values; it only plots actual .npy arrays from the completed SDK run.
if saved_outputs["summary"].get("is_placeholder"):
    print("UTCI grids are placeholders or missing, so no inline UTCI plot is shown.")
else:
    plot_utci_grids(
        saved_outputs["utci_without_trees_npy"],
        saved_outputs["utci_with_trees_npy"],
    )

In [ ]:
# If the live run completed, save interactive Plotly heatmaps for both scenarios.
if result["status"] == "completed":
    without_trees_html = project_root / "outputs" / "utci_without_trees.html"
    with_trees_html = project_root / "outputs" / "utci_with_detected_trees.html"

    saved_without = save_utci_heatmap(
        result["result_without_trees"],
        str(without_trees_html),
        "UTCI without detected trees",
    )
    saved_with = save_utci_heatmap(
        result["result_with_trees"],
        str(with_trees_html),
        "UTCI with detected trees",
    )

    print(f"Saved UTCI heatmap without trees to: {saved_without}")
    print(f"Saved UTCI heatmap with trees to: {saved_with}")
else:
    print("No heatmaps saved because the live UTCI run did not complete.")

In [ ]:
# Print the prepared payload summary.
# This is useful for checking the polygon and vegetation feature counts.
print("Prepared UTCI payload summary:")
pprint(result["payload_summary"])